## **Setup**
-------

In [1]:
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

In [9]:
%pip install google-genai gradio -q

In [10]:
from google import genai
from google.genai import types # types is used for specific configurations later

# Initialize the client with the API key
client = genai.Client(api_key=GOOGLE_API_KEY)

In [11]:
MODEL_ID = "gemini-3.1-flash-lite-preview" # @param["gemini-2.5-flash-lite", "gemini-3.1-flash-lite-preview"] {"allow-input":true, isTemplate: true}

----
## **Test**
----

In [12]:
from IPython.display import Markdown # Used for nice formatting of output
from pydantic import BaseModel # Import Pydantic
# Assume 'client', 'MODEL_ID', 'types' are available

# Define the desired structure using a Pydantic model
class Recipe(BaseModel):
    recipe_name: str
    recipe_description: str
    recipe_ingredients: list[str]

# Make the API call, specifying JSON output and the schema
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Provide a popular cookie recipe and its ingredients.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json", # Request JSON output
        response_schema=Recipe, # Provide the Pydantic model as the schema
    ),
)

# The response.text should now contain a JSON string matching the Recipe schema
# Use Markdown to display it nicely, potentially with JSON formatting
Markdown(f"```json\n{response.text}\n```")
# print(response.text) # Raw JSON string

```json
{
  "recipe_name": "Classic Chocolate Chip Cookies",
  "recipe_description": "A timeless recipe for soft and chewy chocolate chip cookies with a golden brown edge.",
  "recipe_ingredients": [
    "1 cup unsalted butter, softened",
    "3/4 cup granulated sugar",
    "3/4 cup packed brown sugar",
    "2 large eggs",
    "1 teaspoon vanilla extract",
    "2 1/4 cups all-purpose flour",
    "1 teaspoon baking soda",
    "1/2 teaspoon salt",
    "2 cups semi-sweet chocolate chips"
  ]
}
```

## **Explanation of Code**

Let's break down the code step-by-step:

1. **Imports**:

   ```python
   from IPython.display import Markdown
   from pydantic import BaseModel
   ```

   * `IPython.display.Markdown`: This is a module used to render **Markdown** in Jupyter notebooks or IPython environments. Markdown helps display formatted text, such as the JSON content in a human-readable format.
   * `pydantic.BaseModel`: This is part of the **Pydantic** library, which is used for data validation and settings management. Pydantic allows us to define **data models** and automatically validate data that adheres to these models.

2. **Defining the Pydantic Model**:

   ```python
   class Recipe(BaseModel):
       recipe_name: str
       recipe_description: str
       recipe_ingredients: list[str]
   ```

   * **Pydantic Model**: Here, the `Recipe` class is defined as a **Pydantic model**. This class will represent the structure of the data that we expect in our response.
   * **Fields and Types**:

     * `recipe_name`: A string that holds the name of the recipe.
     * `recipe_description`: A string describing the recipe.
     * `recipe_ingredients`: A list of strings representing the ingredients for the recipe.
   * Pydantic models automatically enforce type validation. This means that if the data returned doesn't match the expected types (e.g., if `recipe_ingredients` isn't a list of strings), an error will occur.

3. **Making the API Call**:

   ```python
   response = client.models.generate_content(
       model=MODEL_ID,
       contents="Provide a popular cookie recipe and its ingredients.",
       config=types.GenerateContentConfig(
           response_mime_type="application/json",  # Request JSON output
           response_schema=Recipe,  # Provide the Pydantic model as the schema
       ),
   )
   ```

   * **API Call**: The `client.models.generate_content` function is used to call an API (GPT-like model). The `contents` parameter specifies the prompt ("Provide a popular cookie recipe and its ingredients").
   * **Config**:

     * `response_mime_type="application/json"`: This indicates that the response should be in **JSON format**.
     * `response_schema=Recipe`: By passing the `Recipe` Pydantic model, we tell the API that we expect the response to adhere to this structure (i.e., a JSON object with a `recipe_name`, `recipe_description`, and `recipe_ingredients`).
   * **Response**: The response from the API will be in the form of a JSON string that matches the `Recipe` schema. In other words, the API will return a structured JSON response that contains a recipe with the specified fields.

4. **Displaying the Response**:

   ````python
   Markdown(f"```json\n{response.text}\n```")
   ````

   * **Markdown**: This takes the raw JSON string (`response.text`) and formats it inside a code block (with syntax highlighting for JSON). The formatted output will be visually appealing and easier to read.

5. **(Optional) Print Raw Response**:

   ```python
   # print(response.text) # Raw JSON string
   ```

   * If you want to see the raw JSON response, you can uncomment this line, which will print the raw JSON string before it's formatted for display.

### How **Pydantic** Works in This Example:

* **Data Validation**: When the `Recipe` model is passed as the `response_schema`, Pydantic will ensure that the returned data (the API response) matches the expected structure. If the API returns something that doesn't conform to the `Recipe` model (e.g., missing a `recipe_name` or having an invalid data type), Pydantic will raise an error.

* **Serialization/Deserialization**: Pydantic models automatically handle the conversion between Python objects and JSON. When you pass `response.text` to the `Markdown` display, you get the raw JSON, but behind the scenes, Pydantic can also serialize and deserialize the data. This means you can take the JSON data returned from the API and convert it into a Python object (i.e., an instance of the `Recipe` class) for further manipulation if needed.

* **Automatic Data Parsing**: With Pydantic, we don't need to manually validate or parse the JSON response. The `Recipe` class does this for us. Pydantic handles the complex task of ensuring the data matches the expected types and structure, simplifying the process of working with JSON.

### How It Results in JSON:

The API call requests a response in **JSON format** with the `response_mime_type="application/json"` parameter.

The response from the API will then look something like this:

```json
{
  "recipe_name": "Classic Chocolate Chip Cookies",
  "recipe_description": "A timeless recipe for chewy, soft-centered chocolate chip cookies with crispy golden edges.",
  "recipe_ingredients": [
    "1 cup unsalted butter, softened",
    "3/4 cup granulated sugar",
    "3/4 cup packed brown sugar",
    "2 large eggs",
    "1 teaspoon vanilla extract",
    "2 1/4 cups all-purpose flour",
    "1 teaspoon baking soda",
    "1/2 teaspoon salt",
    "2 cups semi-sweet chocolate chips"
  ]
}
```

This JSON string matches the `Recipe` schema:

* It has a `recipe_name` (string).
* It has a `recipe_description` (string).
* It has `recipe_ingredients`, which is a list of strings.

### Summary:

* **Pydantic** is used here to define the expected structure of the data (`Recipe` model) and automatically validates that the returned data from the API matches this structure.
* The **API response** is returned as JSON, which matches the schema defined by the Pydantic model.
* **Markdown** is used to format and display the JSON nicely in a Jupyter/IPython environment.


